In [0]:
-- =====================================================================
-- mkt_dashboard_top_films_monthly
-- Grain: report_month x rank (1-5, ranked by CC ticket attendance)
-- Shows CC / Scene / Total_Biz attendance for the same top-5 titles
-- Hardcoded to July 2026
-- =====================================================================

WITH params AS (
    SELECT
        DATE('2026-07-01')                               AS month_start,
        LAST_DAY(DATE('2026-07-01'))                     AS month_end
),

-- ---------------------------------------------------------------------
-- Ticket lines at Cineplex Theatre for the month, tagged by population
-- ---------------------------------------------------------------------
ticket_lines AS (
    SELECT
        t.film_id,
        t.Quantity,
        CASE WHEN t.CineClubFLAG = 'Yes' THEN 1 ELSE 0 END AS is_cc,
        CASE WHEN t.SCENEFLAG = 'Yes' AND t.CineClubFLAG IS DISTINCT FROM 'Yes' THEN 1 ELSE 0 END AS is_scene
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
    JOIN cpx_dataanalytics_gold.customer360.cpx_d_location loc
        ON t.LocationId = loc.LocationID
    CROSS JOIN params p
    WHERE loc.LC_Location_Type_Description = 'Cineplex Theatre'
      AND t.TicketFLAG = 'Yes'
      AND t.VisitDateId BETWEEN CAST(DATE_FORMAT(p.month_start, 'yyyyMMdd') AS INT)
                             AND CAST(DATE_FORMAT(p.month_end, 'yyyyMMdd') AS INT)
),

-- ---------------------------------------------------------------------
-- Per-film attendance by population, rolled up to title_name
-- (title_name groups all format/language variants of a film together)
-- ---------------------------------------------------------------------
film_attendance AS (
    SELECT
        f.title_name,
        SUM(CASE WHEN tl.is_cc = 1 THEN tl.Quantity ELSE 0 END)       AS cc_tickets,
        SUM(CASE WHEN tl.is_scene = 1 THEN tl.Quantity ELSE 0 END)    AS scene_tickets,
        SUM(tl.Quantity)                                              AS total_biz_tickets
    FROM ticket_lines tl
    JOIN cpx_dataanalytics_gold.customer360.cpx_d_film f
        ON tl.film_id = f.film_id
    GROUP BY f.title_name
),

-- ---------------------------------------------------------------------
-- Top 5 titles ranked by CC attendance
-- ---------------------------------------------------------------------
ranked AS (
    SELECT
        title_name,
        cc_tickets,
        scene_tickets,
        total_biz_tickets,
        ROW_NUMBER() OVER (ORDER BY cc_tickets DESC) AS rnk
    FROM film_attendance
)

SELECT
    p.month_start   AS report_month,
    r.rnk           AS rank,
    r.title_name,
    r.cc_tickets,
    r.scene_tickets,
    r.total_biz_tickets
FROM ranked r
CROSS JOIN params p
WHERE r.rnk <= 5
ORDER BY r.rnk;